In [4]:
"""
Statistical Exploration: Correlation, Conditional Probabilities, and Independence
Using matplotlib, seaborn, and scipy.stats
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, pearsonr, spearmanr, kendalltau

# ─────────────────────────────────────────────
# GLOBAL STYLE
# ─────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f14",
    "axes.facecolor": "#16161d",
    "axes.edgecolor": "#3a3a50",
    "axes.labelcolor": "#c8c8e0",
    "axes.titlecolor": "#e8e8ff",
    "xtick.color": "#888899",
    "ytick.color": "#888899",
    "text.color": "#c8c8e0",
    "grid.color": "#2a2a3a",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
    "font.family": "monospace",
})

ACCENT1 = "#7f6fff"   # violet
ACCENT2 = "#ff6f9f"   # pink
ACCENT3 = "#6fffbf"   # mint
ACCENT4 = "#ffbf6f"   # amber
PALETTE = [ACCENT1, ACCENT2, ACCENT3, ACCENT4]

np.random.seed(42)

# ─────────────────────────────────────────────
# SECTION 1: CORRELATION
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("  SECTION 1: CORRELATION")
print("="*60)

n = 300

# Four datasets with different correlation structures
x = np.random.normal(0, 1, n)
y_strong_pos  = 0.9 * x + 0.1 * np.random.normal(0, 1, n)   # r ≈ +0.9
y_weak_pos    = 0.3 * x + 0.7 * np.random.normal(0, 1, n)   # r ≈ +0.3
y_strong_neg  = -0.9 * x + 0.1 * np.random.normal(0, 1, n)  # r ≈ -0.9
y_none        = np.random.normal(0, 1, n)                     # r ≈  0
y_nonlinear   = x**2 + 0.2 * np.random.normal(0, 1, n)       # non-linear

datasets = [
    (x, y_strong_pos, "Strong Positive\nr ≈ +0.9"),
    (x, y_weak_pos,   "Weak Positive\nr ≈ +0.3"),
    (x, y_strong_neg, "Strong Negative\nr ≈ −0.9"),
    (x, y_none,       "No Correlation\nr ≈ 0"),
    (x, y_nonlinear,  "Non-linear\n(Pearson misleads)"),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle("CORRELATION PATTERNS", fontsize=15, fontweight="bold",
             color="#e8e8ff", y=1.02)

for ax, (xi, yi, label), color in zip(axes, datasets, PALETTE + [ACCENT1]):
    pr, pp = pearsonr(xi, yi)
    sr, sp = spearmanr(xi, yi)

    ax.scatter(xi, yi, alpha=0.4, s=12, color=color)
    # regression line
    m, b = np.polyfit(xi, yi, 1)
    xlim = np.array([xi.min(), xi.max()])
    ax.plot(xlim, m*xlim + b, color=color, linewidth=2, alpha=0.9)

    ax.set_title(label, fontsize=9, color="#e8e8ff")
    ax.text(0.05, 0.95,
            f"Pearson r = {pr:+.3f}\nSpearman ρ = {sr:+.3f}",
            transform=ax.transAxes, fontsize=8,
            verticalalignment="top", color=ACCENT3,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#1e1e2e", alpha=0.8))
    ax.grid(True)
    ax.set_xlabel("X"); ax.set_ylabel("Y")

plt.tight_layout()
plt.savefig("outputs/1_correlation_patterns.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f14")
plt.close()
print("  ✓ Saved: 1_correlation_patterns.png")

# ── Correlation Matrix ──────────────────────────────────────
print("\n--- Correlation Matrix ---")
df = pd.DataFrame({
    "Income":       np.random.normal(50000, 15000, n),
    "Education":    np.random.randint(10, 22, n).astype(float),
    "Age":          np.random.normal(40, 12, n),
    "Hours/Week":   np.random.normal(40, 8, n),
})
df["Savings"] = 0.7 * df["Income"] + 0.3 * df["Education"] * 2000 + np.random.normal(0, 5000, n)
df["Debt"]    = -0.4 * df["Income"] + 0.3 * df["Age"] * 500 + np.random.normal(0, 8000, n)

corr = df.corr()
print(corr.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CORRELATION MATRIX ANALYSIS", fontsize=13, fontweight="bold", color="#e8e8ff")

# Pearson heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(260, 330, s=80, l=45, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, annot=True, fmt=".2f",
            square=True, linewidths=0.5, ax=axes[0],
            cbar_kws={"shrink": .7},
            annot_kws={"size": 11, "color": "white"})
axes[0].set_title("Pearson Correlation Matrix\n(lower triangle)", color="#e8e8ff")

# Scatter matrix (pairplot-style using seaborn on axes[1] as placeholder text)
axes[1].axis("off")
axes[1].text(0.5, 0.6, "Pairplot saved separately\n(see 1b_pairplot.png)",
             ha="center", va="center", color=ACCENT3, fontsize=11)
axes[1].set_facecolor("#16161d")

plt.tight_layout()
plt.savefig("outputs/1b_correlation_matrix.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f14")
plt.close()

# Pairplot
g = sns.pairplot(df, diag_kind="kde", plot_kws={"alpha": 0.4, "color": ACCENT1},
                 diag_kws={"color": ACCENT2, "fill": True, "alpha": 0.5})
g.figure.set_facecolor("#0f0f14")
for ax in g.axes.flatten():
    if ax:
        ax.set_facecolor("#16161d")
        ax.tick_params(colors="#888899")
        for spine in ax.spines.values():
            spine.set_edgecolor("#3a3a50")
g.figure.suptitle("Pairplot (scatter + KDE)", y=1.02, color="#e8e8ff", fontsize=12)
plt.savefig("outputs/1c_pairplot.png", dpi=130, bbox_inches="tight",
            facecolor="#0f0f14")
plt.close()
print("  ✓ Saved: 1b_correlation_matrix.png, 1c_pairplot.png")

# ── Hypothesis test for correlation ─────────────────────────
print("\n--- scipy.stats Correlation Tests ---")
r, p = pearsonr(df["Income"], df["Savings"])
print(f"  Pearson  r(Income, Savings) = {r:.4f},  p = {p:.2e}")
rho, p2 = spearmanr(df["Income"], df["Savings"])
print(f"  Spearman ρ(Income, Savings) = {rho:.4f}, p = {p2:.2e}")
tau, p3 = kendalltau(df["Income"], df["Savings"])
print(f"  Kendall  τ(Income, Savings) = {tau:.4f}, p = {p3:.2e}")





  SECTION 1: CORRELATION
  ✓ Saved: 1_correlation_patterns.png

--- Correlation Matrix ---
            Income  Education    Age  Hours/Week  Savings   Debt
Income       1.000      0.078 -0.051       0.001    0.915 -0.610
Education    0.078      1.000 -0.083       0.043    0.206 -0.037
Age         -0.051     -0.083  1.000       0.070   -0.058  0.197
Hours/Week   0.001      0.043  0.070       1.000    0.047  0.011
Savings      0.915      0.206 -0.058       0.047    1.000 -0.553
Debt        -0.610     -0.037  0.197       0.011   -0.553  1.000
  ✓ Saved: 1b_correlation_matrix.png, 1c_pairplot.png

--- scipy.stats Correlation Tests ---
  Pearson  r(Income, Savings) = 0.9149,  p = 2.10e-119
  Spearman ρ(Income, Savings) = 0.8994, p = 4.35e-109
  Kendall  τ(Income, Savings) = 0.7273, p = 9.95e-79

  SECTION 2: CONDITIONAL PROBABILITIES

Contingency Table (counts):
Job          Admin  Blue Collar  Management  Research  Service  Tech
Education                                                   

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2: CONDITIONAL PROBABILITIES
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("  SECTION 2: CONDITIONAL PROBABILITIES")
print("="*60)

# Build a contingency table: Education Level × Job Category
np.random.seed(7)
education = np.random.choice(["High School", "Bachelor", "Master", "PhD"],
                              size=500, p=[0.35, 0.40, 0.18, 0.07])
# Job depends on education
job_map = {
    "High School": ["Blue Collar", "Service", "Admin", "Tech"],
    "Bachelor":    ["Admin", "Tech", "Management", "Service"],
    "Master":      ["Tech", "Management", "Research", "Admin"],
    "PhD":         ["Research", "Management", "Tech", "Admin"],
}
job_prob = {
    "High School": [0.5,  0.3,  0.15, 0.05],
    "Bachelor":    [0.2,  0.35, 0.25, 0.20],
    "Master":      [0.15, 0.35, 0.35, 0.15],
    "PhD":         [0.55, 0.25, 0.15, 0.05],
}
jobs = [np.random.choice(job_map[e], p=job_prob[e]) for e in education]

ct_df = pd.DataFrame({"Education": education, "Job": jobs})
contingency = pd.crosstab(ct_df["Education"], ct_df["Job"])
print("\nContingency Table (counts):")
print(contingency)

# Joint probabilities P(A ∩ B)
joint = contingency / contingency.values.sum()
print("\nJoint Probability P(Education ∩ Job):")
print(joint.round(4))

# Marginal probabilities
marg_edu = joint.sum(axis=1)
marg_job = joint.sum(axis=0)
print("\nMarginal P(Education):", marg_edu.round(4).to_dict())
print("Marginal P(Job):",       marg_job.round(4).to_dict())

# Conditional P(Job | Education)
cond_job_given_edu = contingency.div(contingency.sum(axis=1), axis=0)
print("\nConditional P(Job | Education):")
print(cond_job_given_edu.round(4))

# Conditional P(Education | Job)
cond_edu_given_job = contingency.div(contingency.sum(axis=0), axis=1)
print("\nConditional P(Education | Job):")
print(cond_edu_given_job.round(4))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("CONDITIONAL PROBABILITIES", fontsize=15, fontweight="bold", color="#e8e8ff")

cmap2 = sns.light_palette(ACCENT1, as_cmap=True)

# 1) Joint probability heatmap
sns.heatmap(joint, annot=True, fmt=".3f", cmap=cmap2, ax=axes[0, 0],
            linewidths=0.5, annot_kws={"size": 9})
axes[0, 0].set_title("Joint Probability P(E ∩ J)", color="#e8e8ff")

# 2) Conditional P(Job | Education) — stacked bar
cond_job_given_edu.plot(kind="bar", stacked=True, ax=axes[0, 1],
                        color=PALETTE, alpha=0.9, edgecolor="#0f0f14")
axes[0, 1].set_title("P(Job | Education) — stacked", color="#e8e8ff")
axes[0, 1].set_xlabel("Education"); axes[0, 1].set_ylabel("Probability")
axes[0, 1].legend(title="Job", bbox_to_anchor=(1, 1), loc="upper left",
                  facecolor="#1e1e2e", labelcolor="#c8c8e0")
axes[0, 1].tick_params(axis="x", rotation=15)

# 3) Conditional P(Education | Job) heatmap
sns.heatmap(cond_edu_given_job.T, annot=True, fmt=".3f",
            cmap=sns.light_palette(ACCENT2, as_cmap=True),
            ax=axes[1, 0], linewidths=0.5, annot_kws={"size": 9})
axes[1, 0].set_title("P(Education | Job)", color="#e8e8ff")

# 4) Bayes' theorem demo
# P(PhD | Research) = P(Research | PhD) * P(PhD) / P(Research)
p_phd = marg_edu["PhD"]
p_research = marg_job["Research"]
p_research_given_phd = cond_job_given_edu.loc["PhD", "Research"]
p_phd_given_research = (p_research_given_phd * p_phd) / p_research

axes[1, 1].axis("off")
bayes_text = (
    "BAYES' THEOREM DEMO\n\n"
    "P(PhD | Research) = ?\n\n"
    f"  P(Research | PhD) = {p_research_given_phd:.4f}\n"
    f"  P(PhD)            = {p_phd:.4f}\n"
    f"  P(Research)       = {p_research:.4f}\n\n"
    "P(PhD|Research) =\n"
    "  P(R|PhD)·P(PhD) / P(R)\n"
    f"= {p_research_given_phd:.4f} × {p_phd:.4f} / {p_research:.4f}\n"
    f"= {p_phd_given_research:.4f}"
)
axes[1, 1].text(0.1, 0.9, bayes_text, transform=axes[1, 1].transAxes,
                fontsize=11, verticalalignment="top", color=ACCENT3,
                fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.6", facecolor="#1a1a2e", alpha=0.9))
axes[1, 1].set_facecolor("#16161d")
axes[1, 1].set_title("Bayes' Theorem", color="#e8e8ff")

for ax in axes.flat:
    if ax.get_title():
        ax.title.set_color("#e8e8ff")

plt.tight_layout()
plt.savefig("outputs/2_conditional_probabilities.png", dpi=150,
            bbox_inches="tight", facecolor="#0f0f14")
plt.close()
print("  ✓ Saved: 2_conditional_probabilities.png")





In [ ]:
# ─────────────────────────────────────────────
# SECTION 3: INDEPENDENCE TESTING
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("  SECTION 3: STATISTICAL INDEPENDENCE")
print("="*60)

# ── Chi-Square Test of Independence ─────────────────────────
print("\n--- Chi-Square Test of Independence ---")

def run_chi2(table, label):
    chi2, p, dof, expected = chi2_contingency(table)
    print(f"\n  [{label}]")
    print(f"    χ² = {chi2:.4f},  p = {p:.4e},  df = {dof}")
    if p < 0.05:
        print(f"    → REJECT H₀: variables are NOT independent (p < 0.05)")
    else:
        print(f"    → FAIL TO REJECT H₀: consistent with independence (p ≥ 0.05)")
    # Cramér's V (effect size)
    n_total = table.values.sum()
    k = min(table.shape) - 1
    v = np.sqrt(chi2 / (n_total * k))
    print(f"    Cramér's V = {v:.4f} (effect size)")
    return chi2, p, dof, expected

# Test 1: Education vs Job (should be dependent — we built them that way)
chi2_a, p_a, dof_a, exp_a = run_chi2(contingency, "Education vs Job (dependent by design)")

# Test 2: Create truly independent variables
group_A = np.random.choice(["A1", "A2", "A3"], size=400, p=[0.4, 0.35, 0.25])
group_B = np.random.choice(["B1", "B2"],        size=400, p=[0.6, 0.4])
indep_table = pd.crosstab(group_A, group_B)
chi2_b, p_b, dof_b, exp_b = run_chi2(indep_table, "Random Groups A vs B (should be independent)")

# ── Visualize independence: Observed vs Expected ─────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("INDEPENDENCE TESTING", fontsize=15, fontweight="bold", color="#e8e8ff")

# Row 0: Education–Job (dependent)
obs_a = contingency.values.astype(float)
exp_a_arr = exp_a

sns.heatmap(obs_a, annot=True, fmt=".0f", ax=axes[0, 0],
            xticklabels=contingency.columns, yticklabels=contingency.index,
            cmap=sns.light_palette(ACCENT1, as_cmap=True), linewidths=0.5)
axes[0, 0].set_title("Observed — Education×Job", color="#e8e8ff")

sns.heatmap(exp_a_arr, annot=True, fmt=".1f", ax=axes[0, 1],
            xticklabels=contingency.columns, yticklabels=contingency.index,
            cmap=sns.light_palette(ACCENT4, as_cmap=True), linewidths=0.5)
axes[0, 1].set_title("Expected (if independent) — Education×Job", color="#e8e8ff")

residuals_a = (obs_a - exp_a_arr) / np.sqrt(exp_a_arr)
sns.heatmap(residuals_a, annot=True, fmt=".2f", ax=axes[0, 2],
            xticklabels=contingency.columns, yticklabels=contingency.index,
            cmap="RdBu_r", center=0, linewidths=0.5)
axes[0, 2].set_title(f"Std. Residuals  |  χ²={chi2_a:.1f}, p={p_a:.2e}", color="#e8e8ff")

# Row 1: Independent groups
obs_b = indep_table.values.astype(float)
exp_b_arr = exp_b

sns.heatmap(obs_b, annot=True, fmt=".0f", ax=axes[1, 0],
            xticklabels=indep_table.columns, yticklabels=indep_table.index,
            cmap=sns.light_palette(ACCENT1, as_cmap=True), linewidths=0.5)
axes[1, 0].set_title("Observed — Random Groups", color="#e8e8ff")

sns.heatmap(exp_b_arr, annot=True, fmt=".1f", ax=axes[1, 1],
            xticklabels=indep_table.columns, yticklabels=indep_table.index,
            cmap=sns.light_palette(ACCENT4, as_cmap=True), linewidths=0.5)
axes[1, 1].set_title("Expected (if independent) — Random Groups", color="#e8e8ff")

residuals_b = (obs_b - exp_b_arr) / np.sqrt(exp_b_arr)
sns.heatmap(residuals_b, annot=True, fmt=".2f", ax=axes[1, 2],
            xticklabels=indep_table.columns, yticklabels=indep_table.index,
            cmap="RdBu_r", center=0, linewidths=0.5)
axes[1, 2].set_title(f"Std. Residuals  |  χ²={chi2_b:.1f}, p={p_b:.2e}", color="#e8e8ff")

for ax in axes.flat:
    ax.tick_params(colors="#888899")

plt.tight_layout()
plt.savefig("outputs/3a_independence_chi2.png", dpi=150,
            bbox_inches="tight", facecolor="#0f0f14")
plt.close()
print("  ✓ Saved: 3a_independence_chi2.png")

# ── Point Biserial & Continuous Independence ────────────────
print("\n--- Continuous Variable Independence ---")

# Correlated pair
xa = np.random.normal(0, 1, 300)
ya_dep = 0.7 * xa + 0.3 * np.random.normal(0, 1, 300)
# Independent pair
xa2 = np.random.normal(0, 1, 300)
ya_indep = np.random.normal(0, 1, 300)

for xi, yi, label in [(xa, ya_dep, "Correlated"), (xa2, ya_indep, "Independent")]:
    r, p = pearsonr(xi, yi)
    stat_ks, p_ks = stats.ks_2samp(xi, yi)
    print(f"  [{label}]  Pearson r={r:.4f} p={p:.4e}  |  KS stat={stat_ks:.4f} p={p_ks:.4e}")

# ── Distribution-based independence: KDE + joint plot ───────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("CONTINUOUS INDEPENDENCE: JOINT DISTRIBUTIONS", fontsize=13,
             fontweight="bold", color="#e8e8ff")

for ax, (xi, yi, label, color) in zip(axes, [
    (xa, ya_dep, "Dependent  (r ≈ 0.7)", ACCENT1),
    (xa2, ya_indep, "Independent  (r ≈ 0)", ACCENT2)
]):
    r, p = pearsonr(xi, yi)
    ax.scatter(xi, yi, alpha=0.25, s=15, color=color)
    sns.kdeplot(x=xi, y=yi, ax=ax, levels=6, color=color, linewidths=1.5, alpha=0.8)
    ax.set_title(f"{label}\nr={r:.3f}, p={p:.2e}", color="#e8e8ff")
    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.grid(True)

plt.tight_layout()
plt.savefig("outputs/3b_independence_continuous.png", dpi=150,
            bbox_inches="tight", facecolor="#0f0f14")
plt.close()
print("  ✓ Saved: 3b_independence_continuous.png")


In [ ]:
# ─────────────────────────────────────────────
# SECTION 4: COMBINED SUMMARY DASHBOARD
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14), facecolor="#0a0a10")
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

fig.suptitle("STATISTICS DASHBOARD: CORRELATION · CONDITIONAL PROB · INDEPENDENCE",
             fontsize=14, fontweight="bold", color="#e8e8ff", y=0.98)

# ─ top row: four correlation scatters ─
titles = ["Strong +", "Weak +", "Strong −", "None"]
ys = [y_strong_pos, y_weak_pos, y_strong_neg, y_none]
for col, (yi, title, color) in enumerate(zip(ys, titles, PALETTE)):
    ax = fig.add_subplot(gs[0, col])
    ax.scatter(x, yi, alpha=0.3, s=8, color=color)
    m, b = np.polyfit(x, yi, 1)
    xlim = np.array([x.min(), x.max()])
    ax.plot(xlim, m*xlim+b, color=color, lw=2)
    r, _ = pearsonr(x, yi)
    ax.set_title(f"Correlation: {title}\nr={r:+.2f}", fontsize=9, color="#e8e8ff")
    ax.grid(True); ax.set_facecolor("#16161d")

# ─ middle row: conditional probs heatmap + stacked bar ─
ax_cond1 = fig.add_subplot(gs[1, :2])
sns.heatmap(cond_job_given_edu, annot=True, fmt=".2f",
            cmap=sns.light_palette(ACCENT3, as_cmap=True),
            ax=ax_cond1, linewidths=0.4, annot_kws={"size": 8})
ax_cond1.set_title("P(Job | Education)", color="#e8e8ff")

ax_cond2 = fig.add_subplot(gs[1, 2:])
cond_job_given_edu.plot(kind="bar", stacked=True, ax=ax_cond2,
                        color=PALETTE, alpha=0.9, edgecolor="#0f0f14")
ax_cond2.set_title("Conditional Probabilities — Stacked Bar", color="#e8e8ff")
ax_cond2.set_facecolor("#16161d")
ax_cond2.tick_params(axis="x", rotation=15)
ax_cond2.legend(facecolor="#1e1e2e", labelcolor="#c8c8e0", fontsize=8)

# ─ bottom row: independence test summary ─
ax_res = fig.add_subplot(gs[2, :2])
sns.heatmap(residuals_a, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            xticklabels=contingency.columns, yticklabels=contingency.index,
            ax=ax_res, linewidths=0.4, annot_kws={"size": 8})
ax_res.set_title(f"Std Residuals: Education×Job  (χ²={chi2_a:.1f}, p={p_a:.2e} → NOT independent)",
                 color="#e8e8ff", fontsize=9)

ax_indep = fig.add_subplot(gs[2, 2:])
ax_indep.scatter(xa, ya_dep, alpha=0.25, s=10, color=ACCENT1, label="Dependent r≈0.7")
ax_indep.scatter(xa2, ya_indep, alpha=0.25, s=10, color=ACCENT2, label="Independent r≈0")
sns.kdeplot(x=xa, y=ya_dep, ax=ax_indep, levels=5, color=ACCENT1, linewidths=1.5, alpha=0.8)
sns.kdeplot(x=xa2, y=ya_indep, ax=ax_indep, levels=5, color=ACCENT2, linewidths=1.5, alpha=0.8)
ax_indep.set_title("Joint KDE: Dependent vs Independent", color="#e8e8ff")
ax_indep.legend(facecolor="#1e1e2e", labelcolor="#c8c8e0", fontsize=9)
ax_indep.set_facecolor("#16161d"); ax_indep.grid(True)

plt.savefig("outputs/4_summary_dashboard.png", dpi=150,
            bbox_inches="tight", facecolor="#0a0a10")
plt.close()
print("\n  ✓ Saved: 4_summary_dashboard.png")

print("\n" + "="*60)
print("  ALL OUTPUTS SAVED TO outputs/")
print("="*60)